In [1]:
import sys
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
from matplotlib.backends.backend_pdf import PdfPages

ROOT = Path("/Users/andreali/Documents/Subgraph_Federated_Learning/")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from andrea.multigraph_generation import TASKS

In [2]:
SELECT_SUBSET_PATH = "clustering"
SELECT_SUBSET = "selected_subset"

EXPERIMENT_LOG_FOLDER = "experiment_log.csv"
DATA_DIR = "clustering/cluster_generation_parameters.csv"

SELECTED_SUBSETS_CSV_PATH = Path(f"./{SELECT_SUBSET_PATH}/{SELECT_SUBSET}.csv")
selected_subset = pd.read_csv(SELECTED_SUBSETS_CSV_PATH)
print("Loaded selected pairs rows:", len(selected_subset))

exp_log = pd.read_csv(EXPERIMENT_LOG_FOLDER)
print("exp rows:", len(exp_log))

test_gen = pd.read_csv(DATA_DIR)
print("Loaded generation rows:", len(test_gen))

Loaded selected pairs rows: 1
exp rows: 72
Loaded generation rows: 170


## 1. Parsing and manifest helpers

In [3]:
def parse_json_list_safe(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, list):
        return x
    return json.loads(x)


def parse_json_dict_safe(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {}
    if isinstance(x, dict):
        return x
    return json.loads(x)


def resolve_project_path(path_like):
    p = Path(path_like)
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.extend(
            [
                p,
                Path.cwd() / p,
                ROOT / p,
            ]
        )
    for cand in candidates:
        if cand.exists():
            return cand.resolve()
    raise FileNotFoundError(f"Could not resolve path: {path_like}")

In [4]:
METHOD_ORDER = ["fedavg", "fedprox"]

METHOD_LABELS = {
    "fedavg": "FedAvg",
    "fedprox": "FedProx",
}

FAMILY_ORDER = [
    "strong_decreasing",
    "mild_decreasing",
    "flat_balanced",
    "mild_increasing",
    "strong_increasing",
]


def family_rank(family_name):
    return FAMILY_ORDER.index(str(family_name))


def load_run_csv(csv_path_like) -> pd.DataFrame:
    csv_path = resolve_project_path(csv_path_like)
    return pd.read_csv(csv_path)


def filter_local_manifest(
    exp_log, subset_clients, model_tag, graph_ids=None, local_epochs=1
):
    part = exp_log[exp_log["run_type"] == "local"].copy()
    part = part[
        (part["subset_clients"] == str(subset_clients))
        & (part["model_tag"] == model_tag)
        & (pd.to_numeric(part["local_epochs"], errors="coerce") == int(local_epochs))
    ].copy()

    if graph_ids is not None:
        graph_ids = {str(g) for g in graph_ids}
        part = part[part["graph_id"].astype(str).isin(graph_ids)].copy()

    part["seed"] = pd.to_numeric(part["seed"], errors="coerce").astype("Int64")
    return part.sort_values(["seed", "graph_id"]).reset_index(drop=True)


def filter_method_manifest(exp_log, subset_clients, model_tag, method, local_epochs=1):
    part = exp_log[exp_log["run_type"] == method].copy()
    part = part[
        (part["subset_clients"] == str(subset_clients))
        & (part["model_tag"] == model_tag)
        & (pd.to_numeric(part["local_epochs"], errors="coerce") == int(local_epochs))
    ].copy()

    part["seed"] = pd.to_numeric(part["seed"], errors="coerce").astype("Int64")
    return part.sort_values(["seed"]).reset_index(drop=True)


def load_local_items(local_rows):
    items = []
    for _, row in local_rows.iterrows():
        items.append(
            {
                "seed": int(row["seed"]),
                "graph_id": str(row["graph_id"]),
                "family": row.get("family", None),
                "local_epochs": int(row["local_epochs"]),
                "df": load_run_csv(row["out_csv"]),
            }
        )
    return items


def load_method_items(method_rows):
    items = []
    for _, row in method_rows.iterrows():
        items.append(
            {
                "seed": int(row["seed"]),
                "local_epochs": int(row["local_epochs"]),
                "method": row["run_type"],
                "df": load_run_csv(row["out_csv"]),
            }
        )
    return items

In [5]:
def _methods_available_for_subset(
    exp_log, subset_clients, model_tag, methods, local_epochs=1
):
    ok = []
    for method in methods:
        part = filter_method_manifest(
            exp_log,
            subset_clients=subset_clients,
            model_tag=model_tag,
            method=method,
            local_epochs=local_epochs,
        )
        if len(part) > 0:
            ok.append(method)
    return ok


def build_global_run_table(
    selected_subset_df, exp_log, methods=METHOD_ORDER, local_epochs=1
):
    rows = []

    local_exp_log = exp_log[exp_log["run_type"] == "local"].copy()

    for _, sel_row in selected_subset_df.iterrows():
        subset_clients = str(sel_row["subset_clients"])
        graph_ids = [str(x) for x in parse_json_list_safe(sel_row["graph_ids_json"])]
        family_order = parse_json_list_safe(sel_row["family_order_json"])
        family_to_graph_ids = {
            str(k): [str(v) for v in vals]
            for k, vals in parse_json_dict_safe(
                sel_row["family_to_graph_ids_json"]
            ).items()
        }
        family_counts = parse_json_dict_safe(sel_row["family_counts_json"])

        candidate_models = sorted(exp_log["model_tag"].dropna().astype(str).unique())

        for model_tag in candidate_models:
            local_rows = filter_local_manifest(
                exp_log,
                subset_clients=subset_clients,
                model_tag=model_tag,
                graph_ids=graph_ids,
                local_epochs=local_epochs,
            )
            if len(local_rows) == 0:
                continue

            available_methods = _methods_available_for_subset(
                exp_log,
                subset_clients=subset_clients,
                model_tag=model_tag,
                methods=methods,
                local_epochs=local_epochs,
            )
            if len(available_methods) == 0:
                continue

            rows.append(
                {
                    "subset_id": sel_row.get("subset_id", None),
                    "subset_clients": subset_clients,
                    "subset_size": int(sel_row.get("subset_size", len(graph_ids))),
                    "graph_ids": graph_ids,
                    "family_order": (
                        family_order if len(family_order) > 0 else FAMILY_ORDER
                    ),
                    "family_to_graph_ids": family_to_graph_ids,
                    "family_counts": family_counts,
                    "model_tag": model_tag,
                    "local_epochs": int(local_epochs),
                    "methods": available_methods,
                }
            )

    return pd.DataFrame(rows)


def build_family_run_table(
    selected_subset_df, exp_log, methods=METHOD_ORDER, local_epochs=1
):
    rows = []

    for _, sel_row in selected_subset_df.iterrows():
        subset_clients = str(sel_row["subset_clients"])
        family_order = parse_json_list_safe(sel_row["family_order_json"])
        family_to_graph_ids = {
            str(k): [str(v) for v in vals]
            for k, vals in parse_json_dict_safe(
                sel_row["family_to_graph_ids_json"]
            ).items()
        }

        candidate_models = sorted(exp_log["model_tag"].dropna().astype(str).unique())

        for model_tag in candidate_models:
            available_methods = _methods_available_for_subset(
                exp_log,
                subset_clients=subset_clients,
                model_tag=model_tag,
                methods=methods,
                local_epochs=local_epochs,
            )
            if len(available_methods) == 0:
                continue

            ordered_families = family_order if len(family_order) > 0 else FAMILY_ORDER

            for fam_idx, family in enumerate(ordered_families):
                graph_ids = family_to_graph_ids.get(str(family), [])
                if not graph_ids:
                    continue

                rows.append(
                    {
                        "subset_id": sel_row.get("subset_id", None),
                        "subset_clients": subset_clients,
                        "family": str(family),
                        "family_order_idx": fam_idx,
                        "subset_size": len(graph_ids),
                        "graph_ids": [str(g) for g in graph_ids],
                        "model_tag": model_tag,
                        "local_epochs": int(local_epochs),
                        "methods": available_methods,
                    }
                )

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["family_order_idx", "model_tag"]).reset_index(drop=True)
    return out


def build_client_run_table(
    selected_subset_df, exp_log, methods=METHOD_ORDER, local_epochs=1
):
    rows = []

    for _, sel_row in selected_subset_df.iterrows():
        subset_clients = str(sel_row["subset_clients"])
        family_order = parse_json_list_safe(sel_row["family_order_json"])
        family_to_graph_ids = {
            str(k): [str(v) for v in vals]
            for k, vals in parse_json_dict_safe(
                sel_row["family_to_graph_ids_json"]
            ).items()
        }
        graph_to_family = {
            str(k): str(v)
            for k, v in parse_json_dict_safe(sel_row["graph_to_family_json"]).items()
        }

        candidate_models = sorted(exp_log["model_tag"].dropna().astype(str).unique())

        for model_tag in candidate_models:
            available_methods = _methods_available_for_subset(
                exp_log,
                subset_clients=subset_clients,
                model_tag=model_tag,
                methods=methods,
                local_epochs=local_epochs,
            )
            if len(available_methods) == 0:
                continue

            ordered_families = family_order if len(family_order) > 0 else FAMILY_ORDER

            for fam_idx, family in enumerate(ordered_families):
                ordered_graph_ids = [
                    str(g) for g in family_to_graph_ids.get(str(family), [])
                ]

                for graph_idx, graph_id in enumerate(ordered_graph_ids):
                    rows.append(
                        {
                            "subset_id": sel_row.get("subset_id", None),
                            "subset_clients": subset_clients,
                            "family": graph_to_family.get(str(graph_id), str(family)),
                            "family_order_idx": fam_idx,
                            "graph_order_idx": graph_idx,
                            "graph_id": str(graph_id),
                            "model_tag": model_tag,
                            "local_epochs": int(local_epochs),
                            "methods": available_methods,
                        }
                    )

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(
            ["family_order_idx", "graph_order_idx", "model_tag"]
        ).reset_index(drop=True)
    return out

In [6]:
def get_group_pooled_task_f1_postagg(
    *,
    method_items,
    task,
    graph_ids,
    split="val",
):
    target_graph_ids = [str(g) for g in graph_ids]
    curves = []

    for item in method_items:
        count_curves = []
        df = item["df"]

        for gid in target_graph_ids:
            count_curve = get_task_count_curve(
                df,
                phase="global_val_client_task",
                split=split,
                task=task,
                graph_id=gid,
            )
            count_curves.append(count_curve)

        pooled = _pool_count_curves(count_curves)
        if pooled.empty:
            continue

        tp = pooled["tp"].to_numpy(dtype=float)
        fp = pooled["fp"].to_numpy(dtype=float)
        fn = pooled["fn"].to_numpy(dtype=float)
        pooled["pooled_f1"] = binary_f1_from_counts(tp, fp, fn)
        curves.append(pooled[["step", "pooled_f1"]])

    return aggregate_seed_curves(curves, "pooled_f1")


def get_weighted_group_loss_for_method(
    *,
    method_items,
    phase,
    graph_ids,
    split="val",
):
    return get_weighted_group_scalar_stats(
        items=method_items,
        phase=phase,
        split=split,
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=True,
    )


def get_pooled_delta_local_minus_method(
    *,
    local_items,
    method_items,
    task,
    graph_ids,
    split="val",
):
    target_graph_ids = {str(g) for g in graph_ids}
    method_by_seed = {int(item["seed"]): item for item in method_items}
    delta_curves = []

    seeds = sorted({int(item["seed"]) for item in local_items})
    for seed in seeds:
        if seed not in method_by_seed:
            continue

        # pooled local
        local_count_curves = []
        seed_local_items = [
            item
            for item in local_items
            if int(item["seed"]) == seed and str(item["graph_id"]) in target_graph_ids
        ]
        for item in seed_local_items:
            count_curve = get_task_count_curve(
                item["df"],
                phase="val_epoch_task",
                split=split,
                task=task,
                graph_id=None,
            )
            local_count_curves.append(count_curve)

        pooled_local = _pool_count_curves(local_count_curves)
        if pooled_local.empty:
            continue
        pooled_local["pooled_f1"] = binary_f1_from_counts(
            pooled_local["tp"].to_numpy(dtype=float),
            pooled_local["fp"].to_numpy(dtype=float),
            pooled_local["fn"].to_numpy(dtype=float),
        )

        # pooled method postagg
        method_df = method_by_seed[seed]["df"]
        method_count_curves = []
        for gid in target_graph_ids:
            count_curve = get_task_count_curve(
                method_df,
                phase="global_val_client_task",
                split=split,
                task=task,
                graph_id=gid,
            )
            method_count_curves.append(count_curve)

        pooled_method = _pool_count_curves(method_count_curves)
        if pooled_method.empty:
            continue
        pooled_method["pooled_f1"] = binary_f1_from_counts(
            pooled_method["tp"].to_numpy(dtype=float),
            pooled_method["fp"].to_numpy(dtype=float),
            pooled_method["fn"].to_numpy(dtype=float),
        )

        merged = pd.merge(
            pooled_local[["step", "pooled_f1"]],
            pooled_method[["step", "pooled_f1"]],
            on="step",
            how="inner",
            suffixes=("_local", "_method"),
        )
        if merged.empty:
            continue

        merged["delta"] = merged["pooled_f1_local"] - merged["pooled_f1_method"]
        delta_curves.append(merged[["step", "delta"]])

    return aggregate_seed_curves(delta_curves, "delta")

In [7]:
def build_overview_global_title(row):
    return (
        f"All-methods overview"
        f" | subset_size={int(row['subset_size'])}"
        f" | methods={', '.join(row['methods'])}"
        f" | model={row['model_tag']}"
        f" | local_epochs={int(row['local_epochs'])}"
    )

def build_overview_family_title(row):
    return (
        f"All-methods family overview"
        f" | family={row['family']}"
        f" | n_clients={int(row['subset_size'])}"
        f" | methods={', '.join(row['methods'])}"
        f" | model={row['model_tag']}"
        f" | local_epochs={int(row['local_epochs'])}"
    )

def build_method_global_title(method, row):
    return (
        f"{METHOD_LABELS[method]} diagnostics"
        f" | global pooled"
        f" | subset_size={int(row['subset_size'])}"
        f" | model={row['model_tag']}"
        f" | local_epochs={int(row['local_epochs'])}"
    )

def build_method_family_title(method, row):
    return (
        f"{METHOD_LABELS[method]} diagnostics"
        f" | family={row['family']}"
        f" | n_clients={int(row['subset_size'])}"
        f" | model={row['model_tag']}"
        f" | local_epochs={int(row['local_epochs'])}"
    )

def build_method_client_title(method, row):
    return (
        f"{METHOD_LABELS[method]} diagnostics"
        f" | family={row['family']}"
        f" | graph_id={row['graph_id']}"
        f" | model={row['model_tag']}"
        f" | local_epochs={int(row['local_epochs'])}"
    )

## 2. Curve utilities

In [8]:
def aggregate_seed_curves(curves, value_col):
    parts = []
    for seed_idx, curve in enumerate(curves):
        if curve is None or curve.empty:
            continue
        part = curve[["step", value_col]].dropna().copy()
        if part.empty:
            continue
        part["seed_idx"] = seed_idx
        parts.append(part)

    if not parts:
        return pd.DataFrame(columns=["step", "mean", "std", "count"])

    full = pd.concat(parts, axis=0, ignore_index=True)
    agg = (
        full.groupby("step")[value_col]
        .agg(["mean", "std", "count"])
        .reset_index()
        .sort_values("step")
    )
    agg["std"] = agg["std"].fillna(0.0)
    return agg


def get_scalar_curve(
    df,
    *,
    phase,
    split,
    metric_col,
    graph_id=None,
):
    part = df[(df["phase"] == phase) & (df["split"] == split)].copy()

    if graph_id is not None and "graph_id" in part.columns:
        part = part[part["graph_id"].astype(str) == str(graph_id)].copy()

    if part.empty or metric_col not in part.columns:
        return pd.DataFrame(columns=["step", metric_col])

    part["step"] = part.apply(_effective_step, axis=1)
    part = part[pd.notna(part["step"])].copy()
    if part.empty:
        return pd.DataFrame(columns=["step", metric_col])

    part["step"] = part["step"].astype(int)
    part[metric_col] = pd.to_numeric(part[metric_col], errors="coerce")
    part = part[pd.notna(part[metric_col])].copy()
    if part.empty:
        return pd.DataFrame(columns=["step", metric_col])

    out = (
        part.groupby("step", as_index=False)[metric_col]
        .mean()
        .sort_values("step")
        .reset_index(drop=True)
    )
    return out


def get_task_curve(
    df,
    *,
    phase,
    split,
    task,
    metric_col,
    graph_id=None,
):
    part = df[
        (df["phase"] == phase) & (df["split"] == split) & (df["task"] == task)
    ].copy()

    if graph_id is not None and "graph_id" in part.columns:
        part = part[part["graph_id"].astype(str) == str(graph_id)].copy()

    if part.empty or metric_col not in part.columns:
        return pd.DataFrame(columns=["step", metric_col])

    part["step"] = part.apply(_effective_step, axis=1)
    part = part[pd.notna(part["step"])].copy()
    if part.empty:
        return pd.DataFrame(columns=["step", metric_col])

    part["step"] = part["step"].astype(int)
    part[metric_col] = pd.to_numeric(part[metric_col], errors="coerce")
    part = part[pd.notna(part[metric_col])].copy()
    if part.empty:
        return pd.DataFrame(columns=["step", metric_col])

    out = (
        part.groupby("step", as_index=False)[metric_col]
        .mean()
        .sort_values("step")
        .reset_index(drop=True)
    )
    return out


def get_task_count_curve(
    df,
    *,
    phase,
    split,
    task,
    graph_id=None,
):
    part = df[
        (df["phase"] == phase) & (df["split"] == split) & (df["task"] == task)
    ].copy()

    if graph_id is not None and "graph_id" in part.columns:
        part = part[part["graph_id"].astype(str) == str(graph_id)].copy()

    needed = ["tp", "fp", "tn", "fn"]
    if part.empty or any(c not in part.columns for c in needed):
        return pd.DataFrame(columns=["step", "tp", "fp", "tn", "fn"])

    part["step"] = part.apply(_effective_step, axis=1)
    part = part[pd.notna(part["step"])].copy()
    if part.empty:
        return pd.DataFrame(columns=["step", "tp", "fp", "tn", "fn"])

    part["step"] = part["step"].astype(int)

    for c in needed:
        part[c] = pd.to_numeric(part[c], errors="coerce")

    part = part.dropna(subset=needed).copy()
    if part.empty:
        return pd.DataFrame(columns=["step", "tp", "fp", "tn", "fn"])

    out = (
        part.groupby("step", as_index=False)[needed]
        .sum()
        .sort_values("step")
        .reset_index(drop=True)
    )
    return out


def binary_f1_from_counts(tp, fp, fn):
    denom = 2.0 * tp + fp + fn
    out = np.full_like(tp, np.nan, dtype=float)
    valid = denom > 0
    out[valid] = (2.0 * tp[valid]) / denom[valid]
    return out


def plot_mean_std(
    ax,
    agg_df,
    *,
    label,
    linestyle="-",
    color=None,
    alpha_fill=0.16,
    linewidth=2.0,
):
    if agg_df is None or agg_df.empty:
        return False

    x = agg_df["step"].to_numpy()
    y = agg_df["mean"].to_numpy()
    s = agg_df["std"].to_numpy()

    (line,) = ax.plot(
        x,
        y,
        label=label,
        linestyle=linestyle,
        color=color,
        linewidth=linewidth,
    )
    fill_color = line.get_color()
    ax.fill_between(x, y - s, y + s, alpha=alpha_fill, color=fill_color)
    return True


def annotate_no_data(ax, text="No data"):
    ax.text(
        0.5,
        0.5,
        text,
        ha="center",
        va="center",
        transform=ax.transAxes,
        fontsize=10,
        color="gray",
    )


def fmt_percent(rate):
    if rate is None or pd.isna(rate):
        return "NA"
    return f"{100.0 * float(rate):.1f}%"


def fmt_count(x):
    if x is None or pd.isna(x):
        return "NA"
    return f"{int(round(float(x)))}"


def fmt_rate_count(rate, count):
    return f"rate={fmt_percent(rate)}, count={fmt_count(count)}"


def _rate_col_from_test_gen(task, test_gen):
    candidates = [
        f"train_{task}_pos_rate",
        f"val_{task}_pos_rate",
        f"test_{task}_pos_rate",
        f"p_{task}",
        task,
    ]
    for c in candidates:
        if c in test_gen.columns:
            return c
    raise KeyError(f"Could not find a rate column for task={task}. Tried: {candidates}")


def _graph_rate_text(graph_id, test_gen):
    part = test_gen[test_gen["graph_id"].astype(str) == str(graph_id)].copy()
    if part.empty:
        return f"graph_id={graph_id} | pos rates: unavailable"

    vals = []
    for task in TASKS:
        col = _rate_col_from_test_gen(task, test_gen)
        v = float(part.iloc[0][col])
        vals.append(f"{task}={100*v:.1f}%")

    return f"graph_id={graph_id} | " + " | ".join(vals)


def _family_rate_text(graph_ids, test_gen, family_name=None):
    part = test_gen[
        test_gen["graph_id"].astype(str).isin([str(g) for g in graph_ids])
    ].copy()
    if part.empty:
        prefix = f"family={family_name} | " if family_name is not None else ""
        return prefix + "pos rates: unavailable"

    vals = []
    for task in TASKS:
        col = _rate_col_from_test_gen(task, test_gen)
        arr = part[col].astype(float).to_numpy()
        vals.append(f"{task}={100*np.mean(arr):.1f}%±{100*np.std(arr):.1f}%")

    prefix = f"family={family_name} | " if family_name is not None else ""
    return prefix + " | ".join(vals)


def _global_family_rate_text(row, test_gen):
    lines = []
    fam_map = row["family_to_graph_ids"]
    fam_order = (
        row["family_order"] if len(row["family_order"]) > 0 else list(fam_map.keys())
    )

    for fam in fam_order:
        gids = fam_map[str(fam)]
        lines.append(_family_rate_text(gids, test_gen, family_name=str(fam)))

    return "\n".join(lines)

## 3. Group aggregation

In [9]:
def _detect_weight_col(df):
    candidates = ["num_nodes", "n_nodes", "node_count", "num_samples"]
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        "Could not find a node-count column in the run CSV. "
        "Expected one of: num_nodes, n_nodes, node_count, num_samples"
    )


def _effective_step(row):
    round_val = row["round"] if "round" in row.index else np.nan
    local_epoch_val = row["local_epoch"] if "local_epoch" in row.index else np.nan

    if pd.notna(round_val):
        return int(round_val)

    if pd.notna(local_epoch_val):
        return int(local_epoch_val)

    return np.nan


def get_weighted_group_scalar_stats(
    *,
    items,
    phase,
    split,
    metric_col,
    graph_ids=None,
    use_graph_filter=True,
):
    seed_curves = []

    for item in items:
        df = item["df"].copy()

        part = df[(df["phase"] == phase) & (df["split"] == split)].copy()
        if part.empty or metric_col not in part.columns:
            continue

        if use_graph_filter and graph_ids is not None and "graph_id" in part.columns:
            part = part[
                part["graph_id"].astype(str).isin([str(g) for g in graph_ids])
            ].copy()

        if part.empty:
            continue

        weight_col = _detect_weight_col(part)

        part["step"] = part.apply(_effective_step, axis=1)
        part = part[pd.notna(part["step"])].copy()
        if part.empty:
            continue

        part["step"] = part["step"].astype(int)

        part[metric_col] = pd.to_numeric(part[metric_col], errors="coerce")
        part[weight_col] = pd.to_numeric(part[weight_col], errors="coerce")

        part = part[
            pd.notna(part[metric_col])
            & pd.notna(part[weight_col])
            & (part[weight_col] > 0)
        ].copy()
        if part.empty:
            continue

        grouped = (
            part.groupby("step", as_index=False)
            .apply(
                lambda g: pd.Series(
                    {
                        metric_col: np.average(
                            g[metric_col].to_numpy(dtype=float),
                            weights=g[weight_col].to_numpy(dtype=float),
                        )
                    }
                )
            )
            .reset_index(drop=True)
            .sort_values("step")
        )

        if not grouped.empty:
            seed_curves.append(grouped[["step", metric_col]])

    return aggregate_seed_curves(seed_curves, metric_col)


def _pool_count_curves(curves):
    good = []
    for curve in curves:
        if curve is None or curve.empty:
            continue
        good.append(curve[["step", "tp", "fp", "tn", "fn"]].copy())

    if not good:
        return pd.DataFrame(columns=["step", "tp", "fp", "tn", "fn"])

    full = pd.concat(good, axis=0, ignore_index=True)
    pooled = (
        full.groupby("step")[["tp", "fp", "tn", "fn"]]
        .sum()
        .reset_index()
        .sort_values("step")
    )
    return pooled


def get_group_pooled_task_f1_local(
    *,
    local_items,
    task,
    graph_ids,
    split="val",
):
    target_graph_ids = {str(g) for g in graph_ids}
    curves = []

    seeds = sorted({int(item["seed"]) for item in local_items})

    for seed in seeds:
        count_curves = []
        seed_items = [
            item
            for item in local_items
            if int(item["seed"]) == seed and str(item["graph_id"]) in target_graph_ids
        ]

        for item in seed_items:
            count_curve = get_task_count_curve(
                item["df"],
                phase="val_epoch_task",
                split=split,
                task=task,
                graph_id=None,
            )
            count_curves.append(count_curve)

        pooled = _pool_count_curves(count_curves)
        if pooled.empty:
            continue

        tp = pooled["tp"].to_numpy(dtype=float)
        fp = pooled["fp"].to_numpy(dtype=float)
        fn = pooled["fn"].to_numpy(dtype=float)

        pooled["pooled_f1"] = binary_f1_from_counts(tp, fp, fn)
        curves.append(pooled[["step", "pooled_f1"]])

    return aggregate_seed_curves(curves, "pooled_f1")


def _rate_col_from_test_gen(task, test_gen):
    candidates = [
        f"train_{task}_pos_rate",
        f"val_{task}_pos_rate",
        f"test_{task}_pos_rate",
        f"p_{task}",
        task,
    ]
    for c in candidates:
        if c in test_gen.columns:
            return c
    raise KeyError(
        f"Could not find a rate column for task={task}. " f"Tried: {candidates}"
    )


def _graph_rate_text(graph_id, test_gen):
    part = test_gen[test_gen["graph_id"].astype(str) == str(graph_id)].copy()
    if part.empty:
        return f"graph_id={graph_id} | pos rates: unavailable"

    vals = []
    for task in TASKS:
        col = _rate_col_from_test_gen(task, test_gen)
        v = float(part.iloc[0][col])
        vals.append(f"{task}={100*v:.1f}%")

    return f"graph_id={graph_id} | " + " | ".join(vals)


def _family_rate_text(graph_ids, test_gen, family_name=None):
    part = test_gen[
        test_gen["graph_id"].astype(str).isin([str(g) for g in graph_ids])
    ].copy()
    if part.empty:
        prefix = f"family={family_name} | " if family_name is not None else ""
        return prefix + "pos rates: unavailable"

    vals = []
    for task in TASKS:
        col = _rate_col_from_test_gen(task, test_gen)
        arr = part[col].astype(float).to_numpy()
        vals.append(f"{task}={100*np.mean(arr):.1f}%±{100*np.std(arr):.1f}%")

    prefix = f"family={family_name} | " if family_name is not None else ""
    return prefix + " | ".join(vals)


def _global_family_rate_text(row, test_gen):
    lines = []
    fam_map = row["family_to_graph_ids"]
    fam_order = (
        row["family_order"] if len(row["family_order"]) > 0 else list(fam_map.keys())
    )

    for fam in fam_order:
        gids = fam_map[str(fam)]
        lines.append(_family_rate_text(gids, test_gen, family_name=str(fam)))

    return "\n".join(lines)

## 4. Plot panels

In [10]:
def plot_overview_loss_panel(
    ax, *, local_items, fedavg_items, fedprox_items, graph_ids
):
    ok = False

    local_agg = get_weighted_group_scalar_stats(
        items=local_items,
        phase="val_epoch",
        split="val",
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=False,
    )
    ok |= plot_mean_std(ax, local_agg, label="pooled standalone val")

    fedavg_post = get_weighted_group_loss_for_method(
        method_items=fedavg_items,
        phase="global_val_client",
        graph_ids=graph_ids,
    )
    fedprox_post = get_weighted_group_loss_for_method(
        method_items=fedprox_items,
        phase="global_val_client",
        graph_ids=graph_ids,
    )

    ok |= plot_mean_std(ax, fedavg_post, label="FedAvg post-aggregation")
    ok |= plot_mean_std(ax, fedprox_post, label="FedProx post-aggregation")

    ax.set_title("validation loss comparison")
    ax.set_xlabel("round")
    ax.set_ylabel("eval_loss")
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8)
    else:
        annotate_no_data(ax)


def plot_overview_local_f1_panel(ax, *, local_items, graph_ids):
    ok = False
    for task in TASKS:
        agg = get_group_pooled_task_f1_local(
            local_items=local_items,
            task=task,
            graph_ids=graph_ids,
        )
        ok |= plot_mean_std(ax, agg, label=task)

    ax.set_title("pooled standalone positive F1")
    ax.set_xlabel("round")
    ax.set_ylabel("positive_f1")
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def plot_overview_method_global_f1_panel(ax, *, method_items, graph_ids, method_name):
    ok = False
    for task in TASKS:
        agg = get_group_pooled_task_f1_postagg(
            method_items=method_items,
            task=task,
            graph_ids=graph_ids,
        )
        ok |= plot_mean_std(ax, agg, label=task)

    ax.set_title(
        f"{METHOD_LABELS[method_name]} pooled global positive F1 (post-aggregation)"
    )
    ax.set_xlabel("round")
    ax.set_ylabel("positive_f1")
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)

In [11]:
def plot_method_loss_panel(ax, *, local_items, method_items, graph_ids, method_name):
    ok = False

    local_agg = get_weighted_group_scalar_stats(
        items=local_items,
        phase="val_epoch",
        split="val",
        metric_col="eval_loss",
        graph_ids=graph_ids,
        use_graph_filter=False,
    )
    post_agg = get_weighted_group_loss_for_method(
        method_items=method_items,
        phase="global_val_client",
        graph_ids=graph_ids,
    )

    ok |= plot_mean_std(ax, local_agg, label="pooled standalone val")
    ok |= plot_mean_std(
        ax, post_agg, label=f"{METHOD_LABELS[method_name]} post-aggregation"
    )

    ax.set_title("validation loss comparison")
    ax.set_xlabel("round")
    ax.set_ylabel("eval_loss")
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8)
    else:
        annotate_no_data(ax)


def plot_method_f1_panel(
    ax, *, local_items, method_items, graph_ids, source, method_name
):
    ok = False

    for task in TASKS:
        if source == "local":
            agg = get_group_pooled_task_f1_local(
                local_items=local_items,
                task=task,
                graph_ids=graph_ids,
            )
            title = "pooled standalone positive F1"
        else:
            agg = get_group_pooled_task_f1_postagg(
                method_items=method_items,
                task=task,
                graph_ids=graph_ids,
            )
            title = (
                f"{METHOD_LABELS[method_name]} pooled positive F1 (post-aggregation)"
            )

        ok |= plot_mean_std(ax, agg, label=task)

    ax.set_title(title)
    ax.set_xlabel("round")
    ax.set_ylabel("positive_f1")
    ax.set_ylim(0.0, 1.0)
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)


def plot_method_delta_panel(ax, *, local_items, method_items, graph_ids, method_name):
    ok = False
    for task in TASKS:
        agg = get_pooled_delta_local_minus_method(
            local_items=local_items,
            method_items=method_items,
            task=task,
            graph_ids=graph_ids,
        )
        ok |= plot_mean_std(ax, agg, label=task)

    ax.axhline(0.0, linestyle="--", linewidth=1, color="black")
    ax.set_title(f"delta = pooled standalone - {METHOD_LABELS[method_name]} global")
    ax.set_xlabel("round")
    ax.set_ylabel("positive_f1 delta")
    ax.grid(alpha=0.3)
    if ok:
        ax.legend(fontsize=8, ncol=2)
    else:
        annotate_no_data(ax)

## 5. Table helpers


In [12]:
BEST_SCALAR_METRICS = [
    ("Subset Acc", "subset_acc", True),
    ("Micro-F1", "micro_f1", True),
    ("Macro Pos-F1", "macro_pos_f1", True),
]


def _best_phase_name(source_kind: str, split: str) -> str:
    prefix = "best_local" if source_kind == "local" else "best_global"
    return f"{prefix}_{split}"


def _select_best_rows(df, *, source_kind, split, graph_ids=None, task=None):
    phase = _best_phase_name(source_kind, split)
    part = df[(df["phase"] == phase) & (df["split"] == split)].copy()

    if graph_ids is not None and "graph_id" in part.columns:
        target = {str(g) for g in graph_ids}
        part = part[part["graph_id"].astype(str).isin(target)].copy()

    if "task" in part.columns:
        if task is None:
            part = part[part["task"].isna()].copy()
        else:
            part = part[part["task"].astype(str) == str(task)].copy()

    return part


def _pool_scalar_rows(part, metric_specs):
    if part is None or part.empty:
        return None

    weight_col = _detect_weight_col(part)
    part = part.copy()
    part[weight_col] = pd.to_numeric(part[weight_col], errors="coerce")
    part = part[pd.notna(part[weight_col]) & (part[weight_col] > 0)].copy()
    if part.empty:
        return None

    out = {
        "num_nodes": float(part[weight_col].sum()),
    }

    weights = part[weight_col].to_numpy(dtype=float)

    for _, col, _ in metric_specs:
        if col not in part.columns:
            out[col] = np.nan
            continue

        vals = pd.to_numeric(part[col], errors="coerce").to_numpy(dtype=float)
        mask = np.isfinite(vals) & np.isfinite(weights) & (weights > 0)

        if not np.any(mask):
            out[col] = np.nan
        else:
            out[col] = float(np.average(vals[mask], weights=weights[mask]))

    return out


def _pool_task_rows(part):
    if part is None or part.empty:
        return None

    part = part.copy()

    # preferred path: compute pooled positive-F1 from pooled tp/fp/fn
    needed = ["tp", "fp", "fn"]
    if all(c in part.columns for c in needed):
        for c in needed:
            part[c] = pd.to_numeric(part[c], errors="coerce")
        part = part.dropna(subset=needed).copy()
        if part.empty:
            return None

        tp = float(part["tp"].sum())
        fp = float(part["fp"].sum())
        fn = float(part["fn"].sum())

        denom = 2.0 * tp + fp + fn
        pos_f1 = np.nan if denom <= 0 else (2.0 * tp) / denom

        return {
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "positive_f1": pos_f1,
        }

    # fallback only if counts are missing
    if "positive_f1" not in part.columns:
        return None

    weight_col = _detect_weight_col(part)
    part[weight_col] = pd.to_numeric(part[weight_col], errors="coerce")
    part["positive_f1"] = pd.to_numeric(part["positive_f1"], errors="coerce")
    part = part[
        pd.notna(part[weight_col])
        & pd.notna(part["positive_f1"])
        & (part[weight_col] > 0)
    ].copy()
    if part.empty:
        return None

    return {
        "positive_f1": float(
            np.average(
                part["positive_f1"].to_numpy(dtype=float),
                weights=part[weight_col].to_numpy(dtype=float),
            )
        )
    }


def collect_best_scalar_by_seed_local(local_items, graph_ids, split="test"):
    target = {str(g) for g in graph_ids}
    by_seed = {}

    for item in local_items:
        graph_id = str(item["graph_id"])
        if graph_id not in target:
            continue

        part = _select_best_rows(
            item["df"],
            source_kind="local",
            split=split,
            graph_ids=[graph_id],
            task=None,
        )
        if part.empty:
            continue

        by_seed.setdefault(int(item["seed"]), []).append(part)

    rows = []
    for seed, frames in sorted(by_seed.items()):
        pooled = _pool_scalar_rows(
            pd.concat(frames, axis=0, ignore_index=True),
            BEST_SCALAR_METRICS,
        )
        if pooled is None:
            continue
        pooled["seed"] = int(seed)
        rows.append(pooled)

    return pd.DataFrame(rows)


def collect_best_scalar_by_seed_method(method_items, graph_ids, split="test"):
    rows = []
    for item in method_items:
        part = _select_best_rows(
            item["df"],
            source_kind="method",
            split=split,
            graph_ids=graph_ids,
            task=None,
        )
        pooled = _pool_scalar_rows(part, BEST_SCALAR_METRICS)
        if pooled is None:
            continue
        pooled["seed"] = int(item["seed"])
        rows.append(pooled)

    return pd.DataFrame(rows)


def collect_best_task_f1_by_seed_local(local_items, graph_ids, task, split="test"):
    target = {str(g) for g in graph_ids}
    by_seed = {}

    for item in local_items:
        graph_id = str(item["graph_id"])
        if graph_id not in target:
            continue

        part = _select_best_rows(
            item["df"],
            source_kind="local",
            split=split,
            graph_ids=[graph_id],
            task=task,
        )
        if part.empty:
            continue

        by_seed.setdefault(int(item["seed"]), []).append(part)

    rows = []
    for seed, frames in sorted(by_seed.items()):
        pooled = _pool_task_rows(pd.concat(frames, axis=0, ignore_index=True))
        if pooled is None:
            continue
        pooled["seed"] = int(seed)
        rows.append(pooled)

    return pd.DataFrame(rows)


def collect_best_task_f1_by_seed_method(method_items, graph_ids, task, split="test"):
    rows = []
    for item in method_items:
        part = _select_best_rows(
            item["df"],
            source_kind="method",
            split=split,
            graph_ids=graph_ids,
            task=task,
        )
        pooled = _pool_task_rows(part)
        if pooled is None:
            continue
        pooled["seed"] = int(item["seed"])
        rows.append(pooled)

    return pd.DataFrame(rows)


def summarize_seed_metric(seed_df, value_col):
    if seed_df is None or seed_df.empty or value_col not in seed_df.columns:
        return {"mean": np.nan, "std": np.nan, "count": 0}

    vals = (
        pd.to_numeric(seed_df[value_col], errors="coerce")
        .dropna()
        .to_numpy(dtype=float)
    )
    if len(vals) == 0:
        return {"mean": np.nan, "std": np.nan, "count": 0}

    return {
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals, ddof=0)),
        "count": int(len(vals)),
    }


def format_metric_summary(stats, *, as_percent=True, digits=1):
    if stats["count"] == 0 or pd.isna(stats["mean"]):
        return "-"

    scale = 100.0 if as_percent else 1.0
    suffix = "%" if as_percent else ""
    return f"{stats['mean'] * scale:.{digits}f}{suffix} ± {stats['std'] * scale:.{digits}f}{suffix}"


def build_overview_algo_table(local_items, method_item_map, graph_ids, split="test"):
    row_map = {
        "Local centralized": {},
    }

    if "fedavg" in method_item_map:
        row_map["FedAvg"] = {}
    if "fedprox" in method_item_map:
        row_map["FedProx"] = {}

    scalar_local = collect_best_scalar_by_seed_local(
        local_items, graph_ids, split=split
    )
    scalar_methods = {
        method: collect_best_scalar_by_seed_method(items, graph_ids, split=split)
        for method, items in method_item_map.items()
    }

    for label, col, as_percent in BEST_SCALAR_METRICS:
        row_map["Local centralized"][label] = format_metric_summary(
            summarize_seed_metric(scalar_local, col),
            as_percent=as_percent,
        )
        for method, method_label in METHOD_LABELS.items():
            if method in scalar_methods and method_label in row_map:
                row_map[method_label][label] = format_metric_summary(
                    summarize_seed_metric(scalar_methods[method], col),
                    as_percent=as_percent,
                )

    for task in TASKS:
        metric_name = f"{task} pos-F1"

        task_local = collect_best_task_f1_by_seed_local(
            local_items, graph_ids, task=task, split=split
        )
        row_map["Local centralized"][metric_name] = format_metric_summary(
            summarize_seed_metric(task_local, "positive_f1"),
            as_percent=True,
        )

        for method, method_label in METHOD_LABELS.items():
            if method in method_item_map and method_label in row_map:
                task_method = collect_best_task_f1_by_seed_method(
                    method_item_map[method], graph_ids, task=task, split=split
                )
                row_map[method_label][metric_name] = format_metric_summary(
                    summarize_seed_metric(task_method, "positive_f1"),
                    as_percent=True,
                )

    columns = ["Algorithm", "Subset Acc", "Micro-F1", "Macro Pos-F1"] + [
        f"{task} pos-F1" for task in TASKS
    ]

    rows = []
    for algo_name, vals in row_map.items():
        row = {"Algorithm": algo_name}
        for c in columns[1:]:
            row[c] = vals.get(c, "-")
        rows.append(row)

    return pd.DataFrame(rows)[columns]


def build_method_algo_table(local_items, method_items, graph_ids, method, split="test"):
    method_label = METHOD_LABELS[method]

    row_map = {
        "Local centralized": {},
        method_label: {},
    }

    scalar_local = collect_best_scalar_by_seed_local(
        local_items, graph_ids, split=split
    )
    scalar_method = collect_best_scalar_by_seed_method(
        method_items, graph_ids, split=split
    )

    for label, col, as_percent in BEST_SCALAR_METRICS:
        row_map["Local centralized"][label] = format_metric_summary(
            summarize_seed_metric(scalar_local, col),
            as_percent=as_percent,
        )
        row_map[method_label][label] = format_metric_summary(
            summarize_seed_metric(scalar_method, col),
            as_percent=as_percent,
        )

    for task in TASKS:
        metric_name = f"{task} pos-F1"

        task_local = collect_best_task_f1_by_seed_local(
            local_items, graph_ids, task=task, split=split
        )
        task_method = collect_best_task_f1_by_seed_method(
            method_items, graph_ids, task=task, split=split
        )

        row_map["Local centralized"][metric_name] = format_metric_summary(
            summarize_seed_metric(task_local, "positive_f1"),
            as_percent=True,
        )
        row_map[method_label][metric_name] = format_metric_summary(
            summarize_seed_metric(task_method, "positive_f1"),
            as_percent=True,
        )

    columns = ["Algorithm", "Subset Acc", "Micro-F1", "Macro Pos-F1"] + [
        f"{task} pos-F1" for task in TASKS
    ]

    rows = []
    for algo_name, vals in row_map.items():
        row = {"Algorithm": algo_name}
        for c in columns[1:]:
            row[c] = vals.get(c, "-")
        rows.append(row)

    return pd.DataFrame(rows)[columns]


def _summary_to_mean(x):
    if not isinstance(x, str):
        return np.nan
    x = x.strip()
    if x in {"-", "", "NA"}:
        return np.nan

    import re

    m = re.match(r"^\s*([-+]?\d+(?:\.\d+)?)\s*%", x)
    if m:
        return float(m.group(1))

    m = re.match(r"^\s*([-+]?\d+(?:\.\d+)?)", x)
    if m:
        return float(m.group(1))

    return np.nan


def draw_algo_table(ax, table_df, title, fontsize=9.0, yscale=1.8):
    ax.axis("off")

    if table_df is None or table_df.empty:
        annotate_no_data(ax, text="No summary table available")
        ax.set_title(title, fontsize=11, pad=10)
        return

    tbl = ax.table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        loc="center",
        cellLoc="center",
        colLoc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(fontsize)
    tbl.scale(1.0, yscale)

    # header + base styling
    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(0.45)
        if r == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#EAEAF2")
        elif c == 0:
            cell.set_text_props(weight="bold")
            cell.get_text().set_color("black")
        else:
            cell.get_text().set_color("#444444")

    # bold best-performing algorithm in each metric column
    for c_idx, col in enumerate(table_df.columns[1:], start=1):
        vals = table_df[col].apply(_summary_to_mean).to_numpy(dtype=float)
        if np.all(np.isnan(vals)):
            continue

        best = np.nanmax(vals)
        for r_idx, val in enumerate(vals, start=1):
            if np.isfinite(val) and np.isclose(val, best):
                tbl[(r_idx, c_idx)].get_text().set_weight("bold")
                tbl[(r_idx, c_idx)].get_text().set_color("black")

    ax.set_title(title, fontsize=11, pad=10)

## 6. Figure builders

In [13]:
def make_overview_global_figure(row, exp_log, test_gen):
    subset_clients = row["subset_clients"]
    model_tag = row["model_tag"]
    graph_ids = [str(g) for g in row["graph_ids"]]

    local_items = load_local_items(
        filter_local_manifest(
            exp_log, subset_clients, model_tag, graph_ids, local_epochs=1
        )
    )
    fedavg_items = load_method_items(
        filter_method_manifest(
            exp_log, subset_clients, model_tag, "fedavg", local_epochs=1
        )
    )
    fedprox_items = load_method_items(
        filter_method_manifest(
            exp_log, subset_clients, model_tag, "fedprox", local_epochs=1
        )
    )

    fig = plt.figure(figsize=(16, 24))
    gs = GridSpec(5, 1, figure=fig, height_ratios=[1.35, 1.0, 1.0, 1.0, 1.0])

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0])
    ax2 = fig.add_subplot(gs[2, 0])
    ax3 = fig.add_subplot(gs[3, 0])
    ax4 = fig.add_subplot(gs[4, 0])

    table_df = build_overview_algo_table(
        local_items=local_items,
        method_item_map={"fedavg": fedavg_items, "fedprox": fedprox_items},
        graph_ids=graph_ids,
        split="test",
    )
    draw_algo_table(ax0, table_df, title="Best test summary (mean ± std over seeds)")

    plot_overview_loss_panel(
        ax1,
        local_items=local_items,
        fedavg_items=fedavg_items,
        fedprox_items=fedprox_items,
        graph_ids=graph_ids,
    )
    plot_overview_local_f1_panel(ax2, local_items=local_items, graph_ids=graph_ids)
    plot_overview_method_global_f1_panel(
        ax3, method_items=fedavg_items, graph_ids=graph_ids, method_name="fedavg"
    )
    plot_overview_method_global_f1_panel(
        ax4, method_items=fedprox_items, graph_ids=graph_ids, method_name="fedprox"
    )

    fig.suptitle(build_overview_global_title(row), fontsize=16, y=0.992)
    fig.text(
        0.5,
        0.962,
        _global_family_rate_text(row, test_gen),
        ha="center",
        va="top",
        fontsize=9,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    return fig


def make_overview_family_figure(row, exp_log, test_gen):
    subset_clients = row["subset_clients"]
    model_tag = row["model_tag"]
    graph_ids = [str(g) for g in row["graph_ids"]]

    local_items = load_local_items(
        filter_local_manifest(
            exp_log, subset_clients, model_tag, graph_ids, local_epochs=1
        )
    )
    fedavg_items = load_method_items(
        filter_method_manifest(
            exp_log, subset_clients, model_tag, "fedavg", local_epochs=1
        )
    )
    fedprox_items = load_method_items(
        filter_method_manifest(
            exp_log, subset_clients, model_tag, "fedprox", local_epochs=1
        )
    )

    fig = plt.figure(figsize=(16, 24))
    gs = GridSpec(5, 1, figure=fig, height_ratios=[1.35, 1.0, 1.0, 1.0, 1.0])

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0])
    ax2 = fig.add_subplot(gs[2, 0])
    ax3 = fig.add_subplot(gs[3, 0])
    ax4 = fig.add_subplot(gs[4, 0])

    table_df = build_overview_algo_table(
        local_items=local_items,
        method_item_map={"fedavg": fedavg_items, "fedprox": fedprox_items},
        graph_ids=graph_ids,
        split="test",
    )
    draw_algo_table(ax0, table_df, title="Best test summary (mean ± std over seeds)")

    plot_overview_loss_panel(
        ax1,
        local_items=local_items,
        fedavg_items=fedavg_items,
        fedprox_items=fedprox_items,
        graph_ids=graph_ids,
    )
    plot_overview_local_f1_panel(ax2, local_items=local_items, graph_ids=graph_ids)
    plot_overview_method_global_f1_panel(
        ax3, method_items=fedavg_items, graph_ids=graph_ids, method_name="fedavg"
    )
    plot_overview_method_global_f1_panel(
        ax4, method_items=fedprox_items, graph_ids=graph_ids, method_name="fedprox"
    )

    fig.suptitle(build_overview_family_title(row), fontsize=16, y=0.992)
    fig.text(
        0.5,
        0.962,
        _family_rate_text(graph_ids, test_gen, family_name=row["family"]),
        ha="center",
        va="top",
        fontsize=9,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    return fig


def make_method_global_figure(method, row, exp_log, test_gen):
    subset_clients = row["subset_clients"]
    model_tag = row["model_tag"]
    graph_ids = [str(g) for g in row["graph_ids"]]

    local_items = load_local_items(
        filter_local_manifest(
            exp_log, subset_clients, model_tag, graph_ids, local_epochs=1
        )
    )
    method_items = load_method_items(
        filter_method_manifest(
            exp_log, subset_clients, model_tag, method, local_epochs=1
        )
    )

    fig = plt.figure(figsize=(16, 24))
    gs = GridSpec(5, 1, figure=fig, height_ratios=[1.35, 1.0, 1.0, 1.0, 1.0])

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0])
    ax2 = fig.add_subplot(gs[2, 0])
    ax3 = fig.add_subplot(gs[3, 0])
    ax4 = fig.add_subplot(gs[4, 0])

    table_df = build_method_algo_table(
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method=method,
        split="test",
    )
    draw_algo_table(
        ax0, table_df, title=f"Best test summary ({METHOD_LABELS[method]} vs Local)"
    )

    plot_method_loss_panel(
        ax1,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_name=method,
    )
    plot_method_f1_panel(
        ax2,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="local",
        method_name=method,
    )
    plot_method_f1_panel(
        ax3,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="postagg",
        method_name=method,
    )
    plot_method_delta_panel(
        ax4,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_name=method,
    )

    fig.suptitle(build_method_global_title(method, row), fontsize=16, y=0.992)
    fig.text(
        0.5,
        0.962,
        _global_family_rate_text(row, test_gen),
        ha="center",
        va="top",
        fontsize=9,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    return fig


def make_method_family_figure(method, row, exp_log, test_gen):
    subset_clients = row["subset_clients"]
    model_tag = row["model_tag"]
    graph_ids = [str(g) for g in row["graph_ids"]]

    local_items = load_local_items(
        filter_local_manifest(
            exp_log, subset_clients, model_tag, graph_ids, local_epochs=1
        )
    )
    method_items = load_method_items(
        filter_method_manifest(
            exp_log, subset_clients, model_tag, method, local_epochs=1
        )
    )

    fig = plt.figure(figsize=(16, 24))
    gs = GridSpec(5, 1, figure=fig, height_ratios=[1.35, 1.0, 1.0, 1.0, 1.0])

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0])
    ax2 = fig.add_subplot(gs[2, 0])
    ax3 = fig.add_subplot(gs[3, 0])
    ax4 = fig.add_subplot(gs[4, 0])

    table_df = build_method_algo_table(
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method=method,
        split="test",
    )
    draw_algo_table(
        ax0, table_df, title=f"Best test summary ({METHOD_LABELS[method]} vs Local)"
    )

    plot_method_loss_panel(
        ax1,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_name=method,
    )
    plot_method_f1_panel(
        ax2,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="local",
        method_name=method,
    )
    plot_method_f1_panel(
        ax3,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="postagg",
        method_name=method,
    )
    plot_method_delta_panel(
        ax4,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_name=method,
    )

    fig.suptitle(build_method_family_title(method, row), fontsize=16, y=0.992)
    fig.text(
        0.5,
        0.962,
        _family_rate_text(graph_ids, test_gen, family_name=row["family"]),
        ha="center",
        va="top",
        fontsize=9,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    return fig


def make_method_client_figure(method, row, exp_log, test_gen):
    subset_clients = row["subset_clients"]
    model_tag = row["model_tag"]
    graph_id = str(row["graph_id"])
    graph_ids = [graph_id]

    local_items = load_local_items(
        filter_local_manifest(
            exp_log, subset_clients, model_tag, graph_ids, local_epochs=1
        )
    )
    method_items = load_method_items(
        filter_method_manifest(
            exp_log, subset_clients, model_tag, method, local_epochs=1
        )
    )

    fig = plt.figure(figsize=(16, 24))
    gs = GridSpec(5, 1, figure=fig, height_ratios=[1.35, 1.0, 1.0, 1.0, 1.0])

    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[1, 0])
    ax2 = fig.add_subplot(gs[2, 0])
    ax3 = fig.add_subplot(gs[3, 0])
    ax4 = fig.add_subplot(gs[4, 0])

    table_df = build_method_algo_table(
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method=method,
        split="test",
    )
    draw_algo_table(
        ax0, table_df, title=f"Best test summary ({METHOD_LABELS[method]} vs Local)"
    )

    plot_method_loss_panel(
        ax1,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_name=method,
    )
    plot_method_f1_panel(
        ax2,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="local",
        method_name=method,
    )
    plot_method_f1_panel(
        ax3,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        source="postagg",
        method_name=method,
    )
    plot_method_delta_panel(
        ax4,
        local_items=local_items,
        method_items=method_items,
        graph_ids=graph_ids,
        method_name=method,
    )

    fig.suptitle(build_method_client_title(method, row), fontsize=16, y=0.992)
    fig.text(
        0.5,
        0.962,
        _graph_rate_text(graph_id, test_gen),
        ha="center",
        va="top",
        fontsize=9,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    return fig

## 7. PDF / combined visual table export


In [14]:
def build_all_methods_overview_pdf(
    global_run_table, family_run_table, exp_log, test_gen, out_path
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    family_sorted = family_run_table.copy()
    if "family_order_idx" in family_sorted.columns:
        family_sorted = family_sorted.sort_values(
            ["family_order_idx", "model_tag"]
        ).reset_index(drop=True)

    with PdfPages(out_path) as pdf:
        for _, row in global_run_table.iterrows():
            fig = make_overview_global_figure(row, exp_log, test_gen)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

        for _, row in family_sorted.iterrows():
            fig = make_overview_family_figure(row, exp_log, test_gen)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

    print(f"saved -> {out_path.resolve()}")


def build_method_diagnostics_pdf(
    method,
    global_run_table,
    family_run_table,
    client_run_table,
    exp_log,
    test_gen,
    out_path,
):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    family_sorted = family_run_table.copy()
    if "family_order_idx" in family_sorted.columns:
        family_sorted = family_sorted.sort_values(
            ["family_order_idx", "model_tag"]
        ).reset_index(drop=True)

    client_sorted = client_run_table.copy()
    sort_cols = [
        c
        for c in ["family_order_idx", "graph_order_idx", "model_tag"]
        if c in client_sorted.columns
    ]
    if len(sort_cols) > 0:
        client_sorted = client_sorted.sort_values(sort_cols).reset_index(drop=True)

    with PdfPages(out_path) as pdf:
        for _, row in global_run_table.iterrows():
            fig = make_method_global_figure(method, row, exp_log, test_gen)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

        for _, row in family_sorted.iterrows():
            fig = make_method_family_figure(method, row, exp_log, test_gen)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

        for _, row in client_sorted.iterrows():
            fig = make_method_client_figure(method, row, exp_log, test_gen)
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

    print(f"saved -> {out_path.resolve()}")

## 8. Runner

In [15]:
# focus only on local_epochs == 1
global_run_table = build_global_run_table(
    selected_subset, exp_log, methods=["fedavg", "fedprox"], local_epochs=1
)
family_run_table = build_family_run_table(
    selected_subset, exp_log, methods=["fedavg", "fedprox"], local_epochs=1
)
client_run_table = build_client_run_table(
    selected_subset, exp_log, methods=["fedavg", "fedprox"], local_epochs=1
)

OUT_DIR = resolve_project_path(".") / "plot_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

build_all_methods_overview_pdf(
    global_run_table=global_run_table,
    family_run_table=family_run_table,
    exp_log=exp_log,
    test_gen=test_gen,
    out_path=OUT_DIR / "all_methods_overview_with_tables.pdf",
)

build_method_diagnostics_pdf(
    method="fedavg",
    global_run_table=global_run_table,
    family_run_table=family_run_table,
    client_run_table=client_run_table,
    exp_log=exp_log,
    test_gen=test_gen,
    out_path=OUT_DIR / "fedavg_diagnostics_with_tables.pdf",
)

build_method_diagnostics_pdf(
    method="fedprox",
    global_run_table=global_run_table,
    family_run_table=family_run_table,
    client_run_table=client_run_table,
    exp_log=exp_log,
    test_gen=test_gen,
    out_path=OUT_DIR / "fedprox_diagnostics_with_tables.pdf",
)

/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_45490/4208458699.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_45490/4208458699.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_45490/4208458699.py:73: DeprecationWarning: DataFrameGroupBy.apply operated o

saved -> /Users/andreali/Documents/Subgraph_Federated_Learning/andrea/plot_outputs/all_methods_overview_with_tables.pdf


/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_45490/4208458699.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_45490/4208458699.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_45490/4208458699.py:73: DeprecationWarning: DataFrameGroupBy.apply operated o

saved -> /Users/andreali/Documents/Subgraph_Federated_Learning/andrea/plot_outputs/fedavg_diagnostics_with_tables.pdf


/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_45490/4208458699.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_45490/4208458699.py:73: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/var/folders/nf/5zqn215x105fjm3njyczpqdm0000gn/T/ipykernel_45490/4208458699.py:73: DeprecationWarning: DataFrameGroupBy.apply operated o

saved -> /Users/andreali/Documents/Subgraph_Federated_Learning/andrea/plot_outputs/fedprox_diagnostics_with_tables.pdf
